# 2.2 Task ETA / Priority Regression — CLO03 (Regression & Feature Engineering)

**Project M.O.N.D.A.Y** · Course 24CSE0316 · Labs 8–13 (Simple, Multiple, and Polynomial Regression).

**Goal.** Predict task completion time (in seconds) given a task's type, step count, target agent, and time-of-day before Tier-2 user confirmation. We formally compare **Simple Linear**, **Multiple Linear**, and **Polynomial Regression (degree=2)**.

**CLO03 evidence produced here:** MAE, MSE, RMSE, R² scores per model, feature interaction evaluation, and prediction error distribution.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from ml.task_eta_regression import (
    generate_action_log,
    engineer_features,
    train_simple_linear,
    train_multiple_linear,
    train_polynomial,
    regression_metrics,
    TASK_TYPES, AGENTS
)

## 1. Load Data & Inspect Distributions

The dataset is synthesized (`datagen/action_logs.py` / `ml/task_eta_regression.py`) to simulate desktop action logs with realistic non-linear step penalties and peak-hour delay multipliers.

In [ ]:
df = generate_action_log(n_samples=2000, seed=42)
print('Dataset size:', len(df))
display(df.head(10))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df['eta_seconds'].hist(ax=ax[0], bins=30, color='skyblue', edgecolor='black')
ax[0].set_title('ETA Target Distribution (seconds)')
ax[0].set_xlabel('ETA (s)')

df.boxplot(column='eta_seconds', by='task_type', ax=ax[1], rot=30)
ax[1].set_title('ETA by Task Type')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Train/Test Split

Features include one-hot task/agent encodings, cyclic time-of-day ($sin/cos$), step count, and historical task averages.

In [ ]:
X, y = engineer_features(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')

## 3. Compare Regression Models (CLO03)

1. **Simple Linear Regression** (step_count feature only)
2. **Multiple Linear Regression** (all engineered features)
3. **Polynomial Regression** (degree=2 polynomial features)

In [ ]:
feature_cols = (
    [f'task_{t}' for t in TASK_TYPES] +
    [f'agent_{a}' for a in AGENTS] +
    ['step_count', 'historical_avg_s', 'tod_sin', 'tod_cos']
)
step_idx = feature_cols.index('step_count')

_, metrics_simple = train_simple_linear(X_train, X_test, y_train, y_test, step_idx)
multiple_pipe, metrics_multiple = train_multiple_linear(X_train, X_test, y_train, y_test)
poly_pipe, metrics_poly = train_polynomial(X_train, X_test, y_train, y_test, degree=2)

results_df = pd.DataFrame([
    {'Model': 'Simple Linear (Step Count)', **metrics_simple},
    {'Model': 'Multiple Linear Regression', **metrics_multiple},
    {'Model': 'Polynomial Regression (deg=2)', **metrics_poly}
]).set_index('Model')

display(results_df)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
results_df['RMSE'].plot(kind='bar', ax=ax[0], color=['salmon', 'orange', 'lightgreen'])
ax[0].set_title('Model Comparison — RMSE (Lower is Better)')
ax[0].set_ylabel('RMSE (seconds)')

results_df['R²'].plot(kind='bar', ax=ax[1], color=['salmon', 'orange', 'lightgreen'])
ax[1].set_title('Model Comparison — R² Score (Higher is Better)')
ax[1].set_ylabel('R² Score')
plt.tight_layout()
plt.show()

## 4. Conclusion & Deployment Winner

Polynomial Regression captures the non-linear interaction between task step counts and time-of-day overheads, yielding the lowest RMSE and highest $R^2$ score.